# HWANGE — Colab driver

This notebook runs the pipeline; it does not contain it. All logic lives in `src/`.
Set the runtime to a **GPU** before running (Runtime > Change runtime type > T4).


## 1. Clone and install

In [ ]:
!git clone https://github.com/semereherruy/TRI-AI-hwange-proj.git
%cd TRI-AI-hwange-proj
!pip install -q -r requirements.txt
!python check_env.py

Cloning into 'TRI-AI-hwange-proj'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 123 (delta 51), reused 105 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 140.54 KiB | 1.04 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/TRI-AI-hwange-proj
Environment
  Python                             3.13.15
  Platform                           Linux x86_64

Dependencies
  PyTorch                            2.11.0+cu128
  Transformers                       5.16.1
  HuggingFace Hub                    1.28.0
  Datasets                           4.0.0
  Accelerate                         1.14.0
  SentencePiece                      0.2.2
  Safetensors                        0.8.0
  pandas                             2.2.3
  NumPy                              2.1.3
  PyArrow                            18.1.0
  scikit-learn                       1.6.1
  SciPy      

## 2. Credentials

Gemma and AfriHate are gated. Add your token to the Colab **Secrets** panel (key icon)
as `HF_TOKEN` — never paste it into a cell.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token loaded:", bool(os.environ.get("HF_TOKEN")))

## 3. Fetch the third-party datasets

The vendored repos are git-ignored, so HateXplain is re-fetched here. AfriHate and
ToxiGen come from the Hub via the loaders.

In [ ]:
!mkdir -p data/raw
!git clone -q --depth 1 https://github.com/hate-alert/HateXplain.git data/raw/HateXplain-master
!ls data/raw/HateXplain-master/Data

## 4. Data pipeline (Phases 2-7)

Each step writes a report. Edit `configs/data.yaml` to change a labelling policy;
every artefact records the policy that produced it.

In [ ]:
!python -m src.data.inspection     # Phase 2: inspection reports
!python -m src.data.canonical      # Phase 4: canonical dataset
!python -m src.data.quality        # Phase 5: quality + leakage report
!python -m src.data.splits         # Phase 6: group-level splits
!python -m src.analysis.baselines  # Phase 7: TF-IDF baselines

## 5. Gemma hidden-state extraction (Phase 8)

Smoke-test the path on a few rows first, then run the full extraction. The model is
frozen and used in inference mode only.

In [ ]:
!python -m src.probing.extraction --splits train test --limit 32 --output /tmp/smoke
!python -m src.probing.extraction --splits train val test probe

## 6. Layer-wise probes and controls (Phases 9-10)

In [ ]:
!python -m src.probing.probes --embeddings data/embeddings

## 7. Layer curve

The primary Phase 9 output: performance as a function of depth, read against the
Phase 7 TF-IDF baseline and the Phase 10 control floors.

In [ ]:
import json
import matplotlib.pyplot as plt
from src.probing.probes import layer_curve

results = json.load(open("reports/phase9_probes/probe_results.json"))
curve = layer_curve(results)

fig, ax = plt.subplots(figsize=(9, 4.5))
for pooling, group in curve.groupby("pooling"):
    ax.plot(group["layer"], group["f1"], marker="o", label=f"probe ({pooling})")

controls = results.get("controls", {})
for name, control in controls.items():
    value = control.get("result", {}).get("test", {}).get("f1")
    if value is not None:
        ax.axhline(value, linestyle="--", linewidth=1, label=f"control: {name}")

ax.set_xlabel("layer")
ax.set_ylabel("F1 (test)")
ax.set_title("Layer-wise probe performance")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

curve